# Chapter 3 &mdash; `nthnumeric`: Random Access into Numeric Order

**Concept 16 of the Chapter 3 decomposition:** *`nthnumeric`: Coding a Numeric-Order Generator*

Get the $N$-th string <b>directly</b> from $N$ &mdash; no predecessors generated. The offset $2^b-1$ is the count of all shorter strings.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter3-Star/Concept-Nthnumeric-Generator/Concept-Nthnumeric-Generator.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Given $N$, produce the $N$-th string with no enumeration:

* $b = \lfloor \log_2(N+1) \rfloor$ places are needed;
* fill them with the binary code for $N - (2^b - 1)$, reading bit 0 as `S[0]` and
  bit 1 as `S[1]`.

The offset $2^b-1$ is **the count of all strings shorter than length $b$** (the sum
$1+2+\cdots+2^{b-1}$), so subtracting it converts a global index into an index
*within* the length group &mdash; exactly the two-level structure of Concept 14.

**Pass a list, not a set**: Jove asserts on this, because a set has no order.

## 2. Definitions

### The function, as shipped

In [ ]:
print("nthnumeric(0)  :", repr(nthnumeric(0,  ['a','b'])))
print("nthnumeric(1)  :", repr(nthnumeric(1,  ['a','b'])))
print("nthnumeric(5)  :", repr(nthnumeric(5,  ['a','b'])))
print("first 16       :", [nthnumeric(i, ['a','b']) for i in range(16)])

### Why the offset is $2^b - 1$

In [ ]:
from math import floor, log
def explain(N):
    if N == 0:
        return "N=0 -> '' (the empty string)"
    b = floor(log(N+1, 2))
    off = 2**b - 1
    return ("N=%-3d  width b=%d  strings shorter than b: %d  index within group: %d"
            % (N, b, off, N - off))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch3&nbsp;15.&nbsp;Formal Definition of Lexicographic Order, and `lexlt`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter3-Star/Concept-Lexlt-Predicate/Concept-Lexlt-Predicate.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3-Star/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;1.&nbsp;DFA Everywhere: Why Finite-State Machines Matter](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter4-DFA/Concept-DFA-Everywhere/Concept-DFA-Everywhere.ipynb)&nbsp;&rarr;

---

## 3. Tests

The book's printed values.

In [ ]:
got = [nthnumeric(i, ['a','b']) for i in range(16)]
want = ['', 'a', 'b', 'aa', 'ab', 'ba', 'bb', 'aaa',
        'aab', 'aba', 'abb', 'baa', 'bab', 'bba', 'bbb', 'aaaa']
print("matches the book :", got == want)
assert got == want

The offset explained for a few indices.

In [ ]:
for N in [0,1,2,3,6,7,14,15]:
    print(explain(N))

**Random access**: index 1000 without generating 999 strings first.

In [ ]:
import time
t0 = time.time(); s = nthnumeric(1000, ['0','1']); t1 = time.time()
print("nthnumeric(1000) =", s, " computed in %.6fs" % (t1-t0))
print("its length :", len(s))

It really is numeric order &mdash; it agrees with a two-level sort.

In [ ]:
mine = sorted(lstar({'0','1'}, 4), key=lambda s:(len(s),s))
jove = [nthnumeric(i, ['0','1']) for i in range(len(mine))]
assert mine == jove
print("agrees with length-then-lex sorting on all %d strings :" % len(mine), True)

The list-vs-set caveat, demonstrated.

In [ ]:
try:
    nthnumeric(3, {'a','b'})       # a SET -- order is not defined
except AssertionError as e:
    print("passing a set raises :", str(e)[:60])
    print("  Use a LIST: the order of the two symbols decides which is bit 0.")

## 4. Exercises


1. Why must the left padding with `S[0]` be there? Remove it and find the first
   index that breaks.
2. Generalise `nthnumeric` to alphabets of size 3.
3. Use `nthnumeric` to build the first 20 test strings for a DFA, as Chapter 5 does.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3-Star/Concept-Nthnumeric-Generator')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')